In [1]:
import os
import pickle
from rdkit import Chem
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('final_total.csv')
df

,CHEMBLID,smiles,ido_ic50,tdo_ic50
0,CHEMBL1098875,O=C1c2cc(F)ccc2-n2c1nc1ccccc1c2=O,23.0,60.0
1,CHEMBL1209728,Cc1c(Br)oc2c1C(=O)C(=O)c1c-2ccc2c1CCCC2(C)C,10000.0,10000.0
2,CHEMBL1276265,O=C1c2ccc(Cl)cc2-n2c1nc1ccccc1c2=O,183.0,224200.0
3,CHEMBL1346056,Oc1ccccc1-c1nc2c3ccccc3c3ccccc3c2[nH]1,11790.0,12250.0
4,CHEMBL139935,O=[N+]([O-])c1cc(F)c2cccnc2c1O,14610.0,30230.0
...,...,...,...,...
755,CHEMBL5221020,O[C@@H]1CCCC[C@H]1CNc1c(Br)ccc2[nH]ncc12,1230.0,11240.0
756,CHEMBL5221102,Brc1cc(NC[C@H]2CCCN2)c2cn[nH]c2c1,56450.0,7450.0
757,CHEMBL578036,CC1(C)CCCc2c1ccc1c2C(=O)C(=O)c2c(CO)coc2-1,10000.0,450.0
758,CHEMBL584991,Nc1nonc1/C(=N/O)Nc1ccc(F)c(Cl)c1,50.0,10000.0


In [3]:
df['mol'] = df['smiles'].apply(lambda x: Chem.MolFromSmiles(x))

In [4]:
df

,CHEMBLID,smiles,ido_ic50,tdo_ic50,mol
0,CHEMBL1098875,O=C1c2cc(F)ccc2-n2c1nc1ccccc1c2=O,23.0,60.0,<rdkit.Chem.rdchem.Mol object at 0x7ad0432b2b90>
1,CHEMBL1209728,Cc1c(Br)oc2c1C(=O)C(=O)c1c-2ccc2c1CCCC2(C)C,10000.0,10000.0,<rdkit.Chem.rdchem.Mol object at 0x7ad0432b2c70>
2,CHEMBL1276265,O=C1c2ccc(Cl)cc2-n2c1nc1ccccc1c2=O,183.0,224200.0,<rdkit.Chem.rdchem.Mol object at 0x7ad0432b2ce0>
3,CHEMBL1346056,Oc1ccccc1-c1nc2c3ccccc3c3ccccc3c2[nH]1,11790.0,12250.0,<rdkit.Chem.rdchem.Mol object at 0x7ad0432b2d50>
4,CHEMBL139935,O=[N+]([O-])c1cc(F)c2cccnc2c1O,14610.0,30230.0,<rdkit.Chem.rdchem.Mol object at 0x7ad0432b2dc0>
...,...,...,...,...,...
755,CHEMBL5221020,O[C@@H]1CCCC[C@H]1CNc1c(Br)ccc2[nH]ncc12,1230.0,11240.0,<rdkit.Chem.rdchem.Mol object at 0x7ad04335f7d0>
756,CHEMBL5221102,Brc1cc(NC[C@H]2CCCN2)c2cn[nH]c2c1,56450.0,7450.0,<rdkit.Chem.rdchem.Mol object at 0x7ad04335f840>
757,CHEMBL578036,CC1(C)CCCc2c1ccc1c2C(=O)C(=O)c2c(CO)coc2-1,10000.0,450.0,<rdkit.Chem.rdchem.Mol object at 0x7ad04335f8b0>
758,CHEMBL584991,Nc1nonc1/C(=N/O)Nc1ccc(F)c(Cl)c1,50.0,10000.0,<rdkit.Chem.rdchem.Mol object at 0x7ad04335f920>


In [5]:
from gensim.models import word2vec
from mol2vec.features import mol2alt_sentence, mol2sentence, MolSentence, DfVec, sentences2vec

In [6]:
w2v_model = word2vec.Word2Vec.load('./mol2vec_models/model_300dim.pkl')

In [7]:
def sentences2vec(sentences, model, unseen=None):

        keys = set(model.wv.key_to_index.keys())
        if unseen is None:
            unseen = np.zeros(model.vector_size, dtype=float)

        vectors = []
        for sentence in sentences:
            vec = np.sum([model.wv.get_vector(word) for word in sentence if word in keys], axis=0)
            if len(vec) == 0:
                vec = unseen
            vectors.append(vec)

        return np.array(vectors)

In [8]:
def generate_word_embedding(smiles_example):

        mol = Chem.MolFromSmiles(smiles_example)
        sentence = MolSentence(mol2alt_sentence(mol, radius=1))
        word_embedding = sentences2vec([sentence], w2v_model)
        result = np.array(word_embedding)

        return result

In [ ]:
#Run this for 100 dim:
np1 = np.zeros((len(df), 100))
for i in range(df.shape[0]):
    we = generate_word_embedding(df['smiles'][i])
    np1[i, :] = we
#print(np1)

In [ ]:
#Run this for 300 dim:
np1 = np.zeros((len(df), 300))
for i in range(df.shape[0]):
    we = generate_word_embedding(df['smiles'][i])
    np1[i, :] = we
#print(np1)

In [10]:
np1.shape

(760, 300)

In [13]:
pd1 = pd.DataFrame(np1)
pd1

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
0,2.085246,-2.038275,-3.187863,8.277987,-1.473328,0.553464,-8.218416,-0.252765,5.976957,4.247883,...,-2.514272,7.614711,9.665213,-0.728455,-6.641990,-3.546622,-2.391847,-2.101916,-9.157832,-0.950201
1,2.164320,-0.599885,-3.328220,1.147567,-0.377009,1.582185,-8.842896,2.590148,0.420797,2.394938,...,-0.297584,9.834462,5.563936,-0.803765,-8.674775,-5.131175,-7.110797,-4.463612,-8.928865,-0.134624
2,1.625464,-2.320905,-3.008997,8.197512,-1.164763,1.174054,-7.366459,-0.087216,5.332006,4.546978,...,-2.660922,7.944535,10.210878,-0.773032,-6.216854,-2.984564,-2.426374,-1.107460,-8.576202,-1.356232
3,1.899230,-2.569787,-3.410282,10.681754,-2.165567,1.730022,-11.107470,0.863943,7.557280,3.846437,...,-2.524517,10.409447,12.572979,-1.090598,-4.617105,-2.671958,-3.982201,-0.524764,-10.007677,1.278735
4,2.579365,-0.462055,-2.360175,3.983515,-2.596048,1.301527,-7.521517,1.303868,3.315001,3.748758,...,-0.815256,6.375582,7.597131,-1.780380,-4.569449,-1.531563,-3.190221,-3.058734,-5.828879,0.447189
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,1.712499,-3.792097,-2.364244,3.363801,-0.166023,-0.001611,-8.855852,0.661475,0.801766,0.544993,...,-0.136141,10.374839,5.031422,-1.267375,-7.197216,-3.841792,-7.744902,-5.256713,-8.899922,-1.219385
756,2.866019,-2.303052,-1.917922,4.422251,-0.717792,-0.029453,-7.384969,-0.278774,2.411067,0.309709,...,1.330587,8.674216,4.500438,-1.143635,-5.071250,-3.963346,-6.209548,-5.659032,-7.639552,-0.926412
757,2.318216,-1.639167,-2.197451,0.764488,0.522354,0.561993,-9.767791,1.646610,1.136305,1.585453,...,-0.579593,9.518148,5.871013,-0.282573,-9.717593,-5.270689,-6.158346,-3.637163,-9.239653,0.744431
758,2.228861,-3.710076,-1.460758,3.618860,1.509769,2.148596,-7.016947,-1.973611,5.845882,3.350458,...,-0.805335,5.618067,11.294125,-1.662612,-2.628362,-1.926153,-5.180820,-1.664173,-6.532691,-0.776828


In [27]:
pd1.to_csv('data_mol2vec_300.csv', index_label=False)

In [28]:
pd.read_csv('data_mol2vec_300.csv')

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
0,-0.312471,-2.405579,-2.802194,6.104644,-3.302747,2.132782,-11.554401,-1.303519,12.841496,1.245251,...,-2.266634,8.313022,12.033699,0.327524,-4.533333,-1.758224,-0.526601,-1.253867,-11.678858,-1.990384
1,-0.398744,-2.074742,-3.210000,5.372430,-3.356797,2.119695,-11.753732,-1.878453,12.627577,1.425519,...,-1.800978,7.858071,12.005699,-0.086628,-4.940817,-1.912289,-1.423539,-0.730991,-11.506533,-1.651215
2,-0.432436,-2.534990,-2.572980,6.523919,-3.243336,2.475069,-11.240848,-1.747506,13.119108,1.029475,...,-2.600777,7.060433,12.061170,-0.033323,-4.091536,-2.751830,0.078313,-0.856142,-12.294048,-2.821765
3,0.117826,-2.385302,-2.632396,5.482557,-2.860797,3.267197,-11.957013,-2.668524,12.145401,1.837132,...,-2.336392,7.320007,12.223616,-0.873083,-5.017231,-2.057368,-0.969517,-0.159194,-12.034309,-2.940481
4,0.306891,-2.873120,-2.853569,5.667333,-3.257311,4.287110,-11.575791,-2.426510,11.651583,2.189703,...,-2.753334,7.765927,13.202133,-1.346794,-5.217676,-1.859936,-1.874806,-0.429656,-12.428425,-3.687359
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,2.655300,-2.842201,-3.777298,6.649707,-0.178579,0.891476,-10.158477,0.691258,8.495137,4.491660,...,-1.564274,10.051888,11.231579,-1.543544,-6.817502,-3.610857,-6.220299,-2.214920,-11.609591,-1.229992
107,4.134258,-3.810573,-4.563393,6.969757,-2.449172,-0.185719,-11.566064,1.361088,11.492760,7.205749,...,-1.848593,10.486750,12.788767,-1.171936,-6.558483,-3.965334,-5.688336,-2.009069,-14.108859,-0.118789
108,2.662954,-3.728014,-4.641705,6.792711,-0.860587,-0.408154,-11.873514,1.816386,8.732901,5.294373,...,0.410567,12.126366,9.590009,-2.103147,-7.181460,-4.813581,-8.407176,-1.752042,-12.308663,-0.406115
109,4.112752,-4.696461,-3.869587,7.731331,-1.079638,0.460639,-12.806609,1.753797,7.830065,5.868233,...,-0.706716,14.438841,10.093640,-3.000480,-9.729472,-4.106588,-8.335432,-2.887178,-12.669396,-1.173635
